# SDC — OCV only, 15 s, 5 × 5 grid

Runs **25 OCV measurements**, each configured for **15 seconds**, using the existing `sdc` package. Calibration, movement, pump control, data files, and progress logs are handled by `Experiment`.

**Hardware safety:** This notebook can command actual printer motion, infusion, and potentiostat operation. Confirm the physical sample origin, calibrated clearance, printer limits, pump settings and device addresses before enabling the final cell. At first, test a single point. Notebook interruption does **not** guarantee hardware shutdown.

In [15]:
from pathlib import Path
from datetime import datetime
from sdc.config import Config
from sdc.core.grid import Grid
from sdc.core.experiment import Experiment

cfg = Config()

# Give this run its own folder to avoid overwriting previous measurements.
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
sample_name = "OCV_Sample01"           # CHANGE for each sample
cfg.data_dir = Path(r"C:\Users\deive\OneDrive - University of Kentucky\HRL\Experimental Campaigns\Electrochemistr") / "SDC_Data" / sample_name / run_id
cfg.file_prefix = f"{sample_name}_{run_id}"

# Only OCV; 15 s per point. These keys match sdc/techniques/ocv.py.
cfg.techniques = ["OCV"]
cfg.ocv_params = {
    "time": 1.0,             # seconds
    "time_interval": 0.5,    # seconds
    "voltage_interval": 1000 # retain your existing large voltage-change threshold
}

# A 5×5 grid at 2 mm spacing spans 8×8 mm from first to last point.
# Coordinates are in the grid frame, converted via cfg.geometry.
cfg.grid = Grid(
    mode="grid", rows=5, columns=5,
    spacing_x=2.0, spacing_y=2.0,
    start_x=0.0, start_y=0.0
)

# Existing procedure: calibrate before starting; infuse at every point.
cfg.calibrate_corners = True
cfg.infusion_every = 1
# Wait 1 second after electrolyte infusion
cfg.droplet_stabilize_seconds = 1.0

# Wait 1 second after printer positioning
cfg.settle_seconds = 1.0
# CONNECTIONS: Defaults come from your project. Verify before running:
# cfg.printer_url = "http://..."
# cfg.biologic_address = "..."
# cfg.biologic_channel = 0
# cfg.pump_port = "COM3"

print("Techniques:", cfg.techniques)
print("OCV duration/point:", cfg.ocv_params["time"], "s")
print("Output:", cfg.data_dir)

Techniques: ['OCV']
OCV duration/point: 1.0 s
Output: C:\Users\deive\OneDrive - University of Kentucky\HRL\Experimental Campaigns\Electrochemistr\SDC_Data\OCV_Sample01\20261023_155210


## Preview the positions — no hardware commands

This lists all 25 grid positions. Check that they fit **within your physical sample after the geometry/origin transformation**. The 8 × 8 mm span below is the distance between outer measurement-point centers, not the footprint including the droplet diameter.

In [18]:
points = cfg.grid.points()
assert len(points) == 25
assert cfg.techniques == ["OCV"]
assert cfg.ocv_params["time"] == 1.0
print(f"{len(points)} positions; at least {len(points)*cfg.ocv_params['time']:.0f} s of OCV acquisition (plus movement, pump, stabilization and calibration).")
for i, (row, col, x, y) in enumerate(points, 1):
    print(f"{i:02d}: row {row+1}, col {col+1} | grid X={x:5.2f}, Y={y:5.2f} mm")

25 positions; at least 25 s of OCV acquisition (plus movement, pump, stabilization and calibration).
01: row 1, col 1 | grid X= 0.00, Y= 0.00 mm
02: row 1, col 2 | grid X= 2.00, Y= 0.00 mm
03: row 1, col 3 | grid X= 4.00, Y= 0.00 mm
04: row 1, col 4 | grid X= 6.00, Y= 0.00 mm
05: row 1, col 5 | grid X= 8.00, Y= 0.00 mm
06: row 2, col 1 | grid X= 0.00, Y= 2.00 mm
07: row 2, col 2 | grid X= 2.00, Y= 2.00 mm
08: row 2, col 3 | grid X= 4.00, Y= 2.00 mm
09: row 2, col 4 | grid X= 6.00, Y= 2.00 mm
10: row 2, col 5 | grid X= 8.00, Y= 2.00 mm
11: row 3, col 1 | grid X= 0.00, Y= 4.00 mm
12: row 3, col 2 | grid X= 2.00, Y= 4.00 mm
13: row 3, col 3 | grid X= 4.00, Y= 4.00 mm
14: row 3, col 4 | grid X= 6.00, Y= 4.00 mm
15: row 3, col 5 | grid X= 8.00, Y= 4.00 mm
16: row 4, col 1 | grid X= 0.00, Y= 6.00 mm
17: row 4, col 2 | grid X= 2.00, Y= 6.00 mm
18: row 4, col 3 | grid X= 4.00, Y= 6.00 mm
19: row 4, col 4 | grid X= 6.00, Y= 6.00 mm
20: row 4, col 5 | grid X= 8.00, Y= 6.00 mm
21: row 5, col 1 | 

## Optional initial single-point validation

To test just the first position, run the following cell **before** starting the experiment; afterward, recreate the `Grid(rows=5, columns=5, ...)` configuration above for the full 25-point run. Do not run the one-point test and the full grid into the same output folder.

In [21]:
# UNCOMMENT to reduce to one measurement for the first hardware check:
# cfg.grid = Grid(mode="single", start_x=0.0, start_y=0.0)
# cfg.data_dir = cfg.data_dir.parent / (run_id + "_single_point_test")
# cfg.file_prefix += "_single"

## Start real measurement — explicitly enable when safe

This calls the controller and will perform four-corner mesh calibration (interactive), then move/infuse/contact/acquire/retract at each position. Use the **Python (SDC Control)** kernel with `sdc-control1`, and ensure no other script is controlling the instruments.

In [ ]:
# CHECK physical clearance, origin, cables, pump fluid volume, and BioLogic state.
# Uncomment only after confirming configuration and machine safety:
experiment = Experiment(cfg)
experiment.run()

[2026-10-23 15:52:24] [SDC] Starting SDC experiment
[2026-10-23 15:52:24] [SDC] Checking printer connection
[2026-10-23 15:52:24] [SDC] Starting mesh calibration

[mesh] Starting 4-point plane calibration.
[mesh] No active bed_mesh profile -- toolhead.position should already match nominal G-code Z.
[mesh] You will be asked to contact each of the 4 corners of the sample in turn.


  [mesh] Corner 1 (e.g. front-left)
  [mesh] Jog the nozzle in Klipper's UI (or Moonraker) to make light
         contact with the sample surface at this corner.
  [mesh] Optional fine-jog here (each press moves 0.050 mm):
         w/s = Z up/down   a/d = X-/X+   q/e = Y-/Y+   ENTER = accept point


  >  


  [mesh] Accepted Corner 1 (e.g. front-left) at X=105.6240 Y=104.4310 Z=109.0580
  [mesh] Retracting to clearance height before next corner ...

  [mesh] Corner 2 (e.g. front-right)
  [mesh] Jog the nozzle in Klipper's UI (or Moonraker) to make light
         contact with the sample surface at this corner.
  [mesh] Optional fine-jog here (each press moves 0.050 mm):
         w/s = Z up/down   a/d = X-/X+   q/e = Y-/Y+   ENTER = accept point


  >  


  [mesh] Accepted Corner 2 (e.g. front-right) at X=121.6240 Y=104.4310 Z=108.9380
  [mesh] Retracting to clearance height before next corner ...

  [mesh] Corner 3 (e.g. back-right)
  [mesh] Jog the nozzle in Klipper's UI (or Moonraker) to make light
         contact with the sample surface at this corner.
  [mesh] Optional fine-jog here (each press moves 0.050 mm):
         w/s = Z up/down   a/d = X-/X+   q/e = Y-/Y+   ENTER = accept point


  >  


  [mesh] Accepted Corner 3 (e.g. back-right) at X=121.6240 Y=120.4310 Z=108.9380
  [mesh] Retracting to clearance height before next corner ...

  [mesh] Corner 4 (e.g. back-left)
  [mesh] Jog the nozzle in Klipper's UI (or Moonraker) to make light
         contact with the sample surface at this corner.
  [mesh] Optional fine-jog here (each press moves 0.050 mm):
         w/s = Z up/down   a/d = X-/X+   q/e = Y-/Y+   ENTER = accept point


  >  


  [mesh] Accepted Corner 4 (e.g. back-left) at X=106.6240 Y=120.4310 Z=108.9380

[mesh] Calibration complete.
[mesh] Fitted plane: z = -0.003992*x + -0.003625*y + 109.830141
[mesh] Max residual across corners: 0.0299 mm

[mesh] Corner points used:
    Corner 1 (e.g. front-left)        X=105.6240  Y=104.4310  Z=109.0580
    Corner 2 (e.g. front-right)       X=121.6240  Y=104.4310  Z=108.9380
    Corner 3 (e.g. back-right)        X=121.6240  Y=120.4310  Z=108.9380
    Corner 4 (e.g. back-left)         X=106.6240  Y=120.4310  Z=108.9380
[2026-10-23 16:00:57] [SDC] Mesh calibration completed
[2026-10-23 16:00:57] [SDC] Experiment metadata and calibration saved
[2026-10-23 16:00:57] [SDC] Grid initialized: 25 measurement points
[2026-10-23 16:00:57] [SDC] Initializing pump
[2026-10-23 16:00:57] [SDC] Point 1/25 | Row=0, Column=0 | X=0.00, Y=0.00
[2026-10-23 16:00:57] [SDC] Moving to measurement position
[2026-10-23 16:01:04] [SDC] Starting electrolyte infusion
[2026-10-23 16:01:07] [SDC] In

## Results

Once complete, results are stored under `cfg.data_dir`: OCV CSVs in `dataocv/`, plus `experiment_metadata.json`, `calibration.json`, `experiment_log.csv`, and `experiment_progress.log`. The OCV-only sequence does **not** run LSV/CV/CA/CP/EIS. OCV duration is the programmed acquisition time; total run time is longer.